# SymTRELLIS Evaluation

This notebook rebuilds four evaluation tables directly from the latest score JSON files in the evaluation workspace. Failed or missing evaluations are not assigned penalty values, and every method within one table is averaged over exactly the same samples.

1. The first main table reports arXiv-style symmetry scores, total fold accuracy, and CD on the exact common eval-success set.
2. The second main table reports symmetry scores and CD on the exact common fold-correct set; total fold accuracy is copied from the common eval-success set so that it remains meaningful.
3. The fold table reports per-fold and total accuracy on the common eval-success set. Reflection-only $S_1$ samples are displayed in the fold-2 column, while correctness is still evaluated using their original internal fold label 1.
4. The ablation table reports every available symmetry score and CD on its own exact common eval-success set.

In [1]:
import json
from pathlib import Path

import pandas as pd

## Configuration

`model_folders` selects the latest Kaolin evaluations. SymTRELLIS uses Lanczos noise rescaling, the legacy sparse-structure mapper, the finetuned neighbor-graph shape mapper, and guidance strength 1.0 at both stages.

In [2]:
dataworkspace_dir = Path("/mnt/scratch/eval_workspace_after_acceptance")

model_folders = {
    "TripoSG": "experiments/score_triposg_seed_43",
    "Hunyuan3D-2.1": "experiments/score_hy3d",
    "TRELLIS 2": "experiments/score_vanilla_trellis2",
    "Closest-point Average": "experiments/score_postprocess_vanilla_trellis2_symmetry_gt_closest_point_average",
    "Sector Replication": "experiments/score_postprocess_vanilla_trellis2_symmetry_gt_sector_replication",
    "Voxel Majority": "experiments/score_postprocess_vanilla_trellis2_symmetry_gt_voxel_majority",
    "Ours": (
        "experiments/score_shape_s114514_ns0.5_nr_lanczos_gs1.0_gd0.3_"
        "m_trellis2_shape_neighbor_graph_finetune_ss_ss_s114514_ns0.5_nr_lanczos_gs1.0_gd0.3_"
        "m_trellis2_sparse_structure_swin3d_legacy_sym_old_pred_vc0_sym_old_pred"
    ),
    "Ours w/ gt-symm": (
        "experiments/score_shape_s114514_ns0.5_nr_lanczos_gs1.0_gd0.3_"
        "m_trellis2_shape_neighbor_graph_finetune_ss_ss_s114514_ns0.5_nr_lanczos_gs1.0_gd0.3_"
        "m_trellis2_sparse_structure_swin3d_legacy_sym_gt_vc0_sym_gt"
    ),
    "Vanilla (TRELLIS.2)": "experiments/score_vanilla_trellis2",
    "Sparse structure only": (
        "experiments/score_shape_s114514_ns0.0_gs0.0_gd0.0_"
        "m_trellis2_shape_neighbor_graph_finetune_ss_ss_s114514_ns0.5_nr_lanczos_gs1.0_gd0.3_"
        "m_trellis2_sparse_structure_swin3d_legacy_sym_gt_vc0_sym_gt"
    ),
    "Sparse structure and shape": (
        "experiments/score_shape_s114514_ns0.5_nr_lanczos_gs1.0_gd0.3_"
        "m_trellis2_shape_neighbor_graph_finetune_ss_ss_s114514_ns0.5_nr_lanczos_gs1.0_gd0.3_"
        "m_trellis2_sparse_structure_swin3d_legacy_sym_gt_vc0_sym_gt"
    ),
}

main_models = [
    "Closest-point Average",
    "Sector Replication",
    "Voxel Majority",
    "TripoSG",
    "Hunyuan3D-2.1",
    "TRELLIS 2",
    "Ours",
    "Ours w/ gt-symm",
]

ablation_models = [
    "Vanilla (TRELLIS.2)",
    "Sparse structure only",
    "Sparse structure and shape",
]

## Data loading

Ground-truth folds are reconstructed from shape filenames using the evaluation workspace point-group convention. Each successful score JSON contributes all continuous and thresholded symmetry metrics.

In [3]:
threshold_suffixes = {
    "0.003": "0003",
    "0.01": "001",
    "0.03": "003",
    "0.05": "005",
    "0.1": "01",
}

symmetry_column_names = {
    "max_sd": "Max SD",
    "mean_sd": "Mean SD",
    "max_bad_rate_0003": "Max Err. @ .003",
    "max_bad_rate_001": "Max Err. @ .01",
    "max_bad_rate_003": "Max Err. @ .03",
    "max_bad_rate_005": "Max Err. @ .05",
    "max_bad_rate_01": "Max Err. @ .1",
    "mean_bad_rate_0003": "Mean Err. @ .003",
    "mean_bad_rate_001": "Mean Err. @ .01",
    "mean_bad_rate_003": "Mean Err. @ .03",
    "mean_bad_rate_005": "Mean Err. @ .05",
    "mean_bad_rate_01": "Mean Err. @ .1",
}
full_symmetry_columns = list(symmetry_column_names)
arxiv_symmetry_columns = [
    "max_sd",
    "mean_sd",
    "max_bad_rate_001",
    "max_bad_rate_003",
    "max_bad_rate_01",
    "mean_bad_rate_001",
    "mean_bad_rate_003",
    "mean_bad_rate_01",
]


def read_shape_folds(dataworkspace_dir):
    polyhedral_folds = {"I": 5, "Ih": 5, "O": 4, "Oh": 4, "T": 3, "Td": 3, "Th": 3}
    rows = []

    for shape_path in sorted((dataworkspace_dir / "shapes").glob("*.glb")):
        shape_id_text, group_text, symmetry_axis = shape_path.stem.split("_")
        symmetry_group = group_text[0].upper() + group_text[1:]

        if symmetry_group in polyhedral_folds:
            symmetry_fold = polyhedral_folds[symmetry_group]
        elif symmetry_group == "S1":
            symmetry_fold = 1
        else:
            order = int("".join(character for character in symmetry_group if character.isdigit()))
            symmetry_fold = order // 2 if symmetry_group.startswith("S") else order

        rows.append(
            {
                "shape_id": int(shape_id_text),
                "symmetry_group": symmetry_group,
                "symmetry_axis": symmetry_axis,
                "gt_fold": symmetry_fold,
            }
        )

    return pd.DataFrame(rows).set_index("shape_id")


def read_trial(trial_name, relative_folder, dataworkspace_dir, metadata, shape_folds):
    rows = []

    for score_path in sorted((dataworkspace_dir / relative_folder).glob("*.json")):
        with score_path.open() as file:
            score = json.load(file)

        row = {
            "trial": trial_name,
            "idx": score_path.stem,
            "score_present": True,
            "eval_success": score["eval_success"],
        }

        if score["eval_success"]:
            symmetry = score["self_symm_result"]
            reconstruction = score["reconstruction_result"]
            row.update(
                {
                    "pred_fold": score["pred_fold"],
                    "max_sd": symmetry["max_sd"],
                    "mean_sd": symmetry["mean_sd"],
                    "cd_recon": reconstruction["cd_recon"],
                }
            )
            for threshold, suffix in threshold_suffixes.items():
                row[f"max_bad_rate_{suffix}"] = symmetry["max_bad_rate"][threshold]
                row[f"mean_bad_rate_{suffix}"] = symmetry["mean_bad_rate"][threshold]

        rows.append(row)

    trial_scores = pd.DataFrame(rows)
    trial_scores = metadata[["idx", "shape_id", "view_id"]].merge(
        trial_scores, on="idx", how="left"
    )
    trial_scores["trial"] = trial_name

    return trial_scores.merge(shape_folds[["gt_fold"]], on="shape_id", how="left")


def common_success_subset(scores, selected_models):
    selected_scores = scores[scores["trial"].isin(selected_models)]
    success_matrix = (
        selected_scores.assign(success=selected_scores["eval_success"].eq(True))
        .pivot(index="idx", columns="trial", values="success")
        .reindex(columns=selected_models, fill_value=False)
        .fillna(False)
    )
    common_ids = success_matrix.index[success_matrix.all(axis=1)]
    common_scores = selected_scores[selected_scores["idx"].isin(common_ids)].copy()
    return common_ids, common_scores


def summarize_scores(subset, selected_models):
    subset = subset.copy()
    subset["fold_accuracy"] = subset["pred_fold"].eq(subset["gt_fold"])
    metric_columns = full_symmetry_columns + ["fold_accuracy", "cd_recon"]
    summary = (
        subset.groupby("trial", sort=False)[metric_columns]
        .mean()
        .reindex(selected_models)
    )
    summary[full_symmetry_columns + ["cd_recon"]] *= 1000
    summary["fold_accuracy"] *= 100
    return summary


def format_quality_table(summary, include_full_symmetry=False, include_fold_accuracy=True, include_cd=True):
    symmetry_columns = full_symmetry_columns if include_full_symmetry else arxiv_symmetry_columns
    columns = list(symmetry_columns)
    if include_fold_accuracy:
        columns.append("fold_accuracy")
    if include_cd:
        columns.append("cd_recon")

    names = dict(symmetry_column_names)
    names.update({"fold_accuracy": "Fold Acc. (%)", "cd_recon": "CD"})
    table = summary[columns].rename(columns=names)
    table.index.name = "Method"
    return table

In [4]:
metadata = pd.read_csv(dataworkspace_dir / "metadata.csv", dtype={"idx": str})
shape_folds = read_shape_folds(dataworkspace_dir)
selected_models = list(dict.fromkeys(main_models + ablation_models))
scores = pd.concat(
    [
        read_trial(
            trial_name,
            model_folders[trial_name],
            dataworkspace_dir,
            metadata,
            shape_folds,
        )
        for trial_name in selected_models
    ],
    ignore_index=True,
)

source_summary = (
    scores.assign(
        json_count=scores["score_present"].eq(True),
        successful=scores["eval_success"].eq(True),
        failed=scores["eval_success"].eq(False),
    )
    .groupby("trial", sort=False)[["json_count", "successful", "failed"]]
    .sum()
    .reindex(selected_models)
)
source_summary.insert(0, "relative_folder", [model_folders[name] for name in selected_models])
source_summary.index.name = "Trial"
source_summary

,relative_folder,json_count,successful,failed
Trial,,,,
Closest-point Average,experiments/score_postprocess_vanilla_trellis2...,2120,2076,44
Sector Replication,experiments/score_postprocess_vanilla_trellis2...,2120,2107,13
Voxel Majority,experiments/score_postprocess_vanilla_trellis2...,2120,2077,43
TripoSG,experiments/score_triposg_seed_43,2120,2100,20
Hunyuan3D-2.1,experiments/score_hy3d,2120,2099,21
TRELLIS 2,experiments/score_vanilla_trellis2,2120,2107,13
Ours,experiments/score_shape_s114514_ns0.5_nr_lancz...,2120,2088,32
Ours w/ gt-symm,experiments/score_shape_s114514_ns0.5_nr_lancz...,2120,2080,40
Vanilla (TRELLIS.2),experiments/score_vanilla_trellis2,2120,2107,13


## Main comparison

All three main tables use the same common eval-success set. Table 2 further restricts symmetry scores and CD to samples for which every displayed method predicts the correct fold. All symmetry scores and CD are reported in $\times 10^3$.

In [5]:
fold_groups = ["2", "3", "4", "5", "6", "7", "8", "9", "10+"]

main_ids, main_common_scores = common_success_subset(scores, main_models)
main_common_scores["fold_correct"] = main_common_scores["pred_fold"].eq(
    main_common_scores["gt_fold"]
)
main_common_scores["fold_group"] = main_common_scores["gt_fold"].astype(int).astype(str)
main_common_scores.loc[main_common_scores["gt_fold"].eq(1), "fold_group"] = "2"
main_common_scores.loc[main_common_scores["gt_fold"].ge(10), "fold_group"] = "10+"

main_eval_success_summary = summarize_scores(main_common_scores, main_models)
main_eval_success_table = format_quality_table(main_eval_success_summary)

print(
    f"Table 1 — common eval-success set: {len(main_ids)}/{len(metadata)} "
    f"({100 * len(main_ids) / len(metadata):.2f}%)."
)
display(main_eval_success_table.style.format("{:.3f}"))

Table 1 — common eval-success set: 1995/2120 (94.10%).


,Max SD,Mean SD,Max Err. @ .01,Max Err. @ .03,Max Err. @ .1,Mean Err. @ .01,Mean Err. @ .03,Mean Err. @ .1,Fold Acc. (%),CD
Method,,,,,,,,,,
Closest-point Average,41.014,23.246,112.019,42.500,9.869,70.062,26.590,6.421,92.782,18.496
Sector Replication,0.188,0.106,2.212,0.466,0.000,0.959,0.163,0.000,99.348,18.084
Voxel Majority,6.009,3.578,125.322,23.844,4.416,63.298,12.241,2.324,99.599,42.894
TripoSG,14.966,9.812,415.168,148.792,11.255,271.311,83.336,6.097,78.446,18.952
Hunyuan3D-2.1,14.185,9.328,368.381,141.909,13.333,242.065,82.022,7.412,73.985,20.320
TRELLIS 2,7.468,4.575,200.025,63.427,5.606,118.417,34.089,2.740,83.960,13.786
Ours,3.353,2.239,54.427,26.611,4.659,39.348,17.965,2.757,83.810,15.328
Ours w/ gt-symm,2.799,1.798,45.357,21.992,3.808,31.632,13.791,2.169,93.534,16.015


In [6]:
main_fold_correct_matrix = (
    main_common_scores.pivot(index="idx", columns="trial", values="fold_correct")
    .reindex(columns=main_models, fill_value=False)
    .fillna(False)
)
main_fold_correct_ids = main_fold_correct_matrix.index[
    main_fold_correct_matrix.all(axis=1)
]
main_fold_correct_scores = main_common_scores[
    main_common_scores["idx"].isin(main_fold_correct_ids)
]
main_fold_correct_summary = summarize_scores(main_fold_correct_scores, main_models)
main_fold_correct_summary["fold_accuracy"] = main_eval_success_summary["fold_accuracy"]
main_fold_correct_table = format_quality_table(main_fold_correct_summary)

print(
    f"Table 2 — common fold-correct set: {len(main_fold_correct_ids)}/{len(metadata)} "
    f"({100 * len(main_fold_correct_ids) / len(metadata):.2f}% of all samples; "
    f"{100 * len(main_fold_correct_ids) / len(main_ids):.2f}% of the common eval-success set). "
    "Fold Acc. is retained from Table 1."
)
display(main_fold_correct_table.style.format("{:.3f}"))

Table 2 — common fold-correct set: 1199/2120 (56.56% of all samples; 60.10% of the common eval-success set). Fold Acc. is retained from Table 1.


,Max SD,Mean SD,Max Err. @ .01,Max Err. @ .03,Max Err. @ .1,Mean Err. @ .01,Mean Err. @ .03,Mean Err. @ .1,Fold Acc. (%),CD
Method,,,,,,,,,,
Closest-point Average,7.352,4.579,52.114,18.786,3.671,40.990,15.498,3.521,92.782,14.423
Sector Replication,0.104,0.073,0.002,0.000,0.000,0.001,0.000,0.000,99.348,16.114
Voxel Majority,4.021,2.841,78.421,11.831,0.891,45.892,6.631,0.439,99.599,42.249
TripoSG,10.517,7.697,294.616,91.296,6.543,207.798,59.843,4.429,78.446,16.491
Hunyuan3D-2.1,8.753,6.182,227.650,72.120,6.834,154.155,46.082,4.395,73.985,16.689
TRELLIS 2,3.672,2.700,95.509,23.347,1.524,65.402,16.900,1.290,83.960,11.602
Ours,1.808,1.463,28.439,11.399,1.596,24.559,10.095,1.340,83.810,12.045
Ours w/ gt-symm,1.405,0.987,20.962,8.361,0.890,15.684,5.697,0.557,93.534,12.823


In [7]:
fold_counts = (
    main_common_scores.drop_duplicates("idx")["fold_group"]
    .value_counts()
    .reindex(fold_groups, fill_value=0)
)
main_fold_accuracy = (
    main_common_scores.pivot_table(
        index="trial",
        columns="fold_group",
        values="fold_correct",
        aggfunc="mean",
    )
    .reindex(index=main_models, columns=fold_groups)
    .mul(100)
)
main_fold_accuracy["Total"] = main_eval_success_summary["fold_accuracy"]
main_fold_accuracy.columns = [
    f"{fold} (n={fold_counts[fold]})" for fold in fold_groups
] + [f"Total (n={len(main_ids)})"]
main_fold_accuracy.index.name = "Method"

print(
    f"Table 3 — fold accuracy on the common eval-success set: "
    f"{len(main_ids)}/{len(metadata)} ({100 * len(main_ids) / len(metadata):.2f}%)."
)
display(main_fold_accuracy.style.format("{:.3f}"))

Table 3 — fold accuracy on the common eval-success set: 1995/2120 (94.10%).


,2 (n=625),3 (n=212),4 (n=274),5 (n=194),6 (n=176),7 (n=70),8 (n=160),9 (n=55),10+ (n=229),Total (n=1995)
Method,,,,,,,,,,
Closest-point Average,99.840,95.755,98.540,87.629,96.023,61.429,96.875,67.273,78.603,92.782
Sector Replication,100.000,99.528,100.000,96.907,100.000,100.000,100.000,100.000,97.380,99.348
Voxel Majority,100.000,99.057,100.000,97.423,100.000,100.000,100.000,100.000,99.563,99.599
TripoSG,99.680,67.925,89.051,66.495,78.977,50.000,81.250,47.273,41.485,78.446
Hunyuan3D-2.1,99.360,63.679,85.401,53.608,71.023,45.714,82.500,20.000,35.808,73.985
TRELLIS 2,99.360,59.906,96.350,67.010,88.636,65.714,95.625,52.727,65.066,83.960
Ours,99.680,56.132,97.810,67.526,88.068,62.857,96.250,52.727,65.066,83.810
Ours w/ gt-symm,100.000,77.830,98.540,82.474,98.864,87.143,98.750,85.455,89.956,93.534


## Ablation

The ablation uses its own common eval-success subset and does not filter by fold correctness. It reports all available symmetry metrics and CD in $\times 10^3$.

In [8]:
ablation_ids, ablation_common_scores = common_success_subset(scores, ablation_models)
ablation_summary = summarize_scores(ablation_common_scores, ablation_models)
ablation_table = format_quality_table(
    ablation_summary,
    include_full_symmetry=True,
    include_fold_accuracy=False,
    include_cd=True,
)

print(
    f"Table 4 — ablation common eval-success set: {len(ablation_ids)}/{len(metadata)} "
    f"({100 * len(ablation_ids) / len(metadata):.2f}%)."
)
display(ablation_table.style.format("{:.3f}"))

Table 4 — ablation common eval-success set: 2073/2120 (97.78%).


,Max SD,Mean SD,Max Err. @ .003,Max Err. @ .01,Max Err. @ .03,Max Err. @ .05,Max Err. @ .1,Mean Err. @ .003,Mean Err. @ .01,Mean Err. @ .03,Mean Err. @ .05,Mean Err. @ .1,CD
Method,,,,,,,,,,,,,
Vanilla (TRELLIS.2),7.827,4.819,396.885,208.361,68.243,29.901,6.015,272.075,124.456,37.094,16.250,2.984,13.903
Sparse structure only,3.820,2.377,247.506,71.554,23.610,13.329,4.160,143.437,42.975,14.526,7.723,2.356,16.190
Sparse structure and shape,2.814,1.799,110.761,45.287,22.113,12.628,3.929,68.521,31.535,13.798,7.400,2.210,16.300
